                # 03 Model Training Lab

                Purpose: pull latest code, run baseline models, train tabular
                Phase 1 models, write model cards and summary predictions, and
                generate calibration diagnostics. This is not a backtest.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Model Configuration


In [ ]:
                from pathlib import Path
                print((Path(PROJECT_ROOT) / "configs" / "models.yaml").read_text())
                


## 5. Run Baseline Models


In [ ]:
baseline_result = run_stage_checked("train_baselines", config_name="models")


## 6. Train Tabular ML Models


In [ ]:
tabular_result = run_stage_checked("train_tabular", config_name="models")


## 7. Run Calibration Diagnostics


In [ ]:
calibration_result = run_stage_checked("calibrate_models", config_name="models")


## 8. Inspect Leaderboards And Calibration


In [ ]:
                from pathlib import Path
                import pandas as pd

                experiment_dir = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID
                print("\nBaseline comparison")
                print(pd.read_csv(experiment_dir / "baseline_comparison.csv"))
                print("\nModel leaderboard")
                print(pd.read_csv(experiment_dir / "model_leaderboard.csv"))
                print("\nCalibration metrics")
                print(pd.read_csv(experiment_dir / "calibration_metrics.csv"))
                


## 9. Confirm Phase Boundary


In [ ]:
                print("Training complete for Phase 1 diagnostics.")
                print("No equity curve, drawdown, position sizing, stop-loss, take-profit, or trade journal was produced.")
                
